# MyGram — Finish Table 2 automatically

Notebook này chỉ chạy hoặc resume ba cấu hình Table 2:

- `table2_zh_en_r030`
- `table2_ja_en_r030`
- `table2_fr_en_r030`

Sáu run Table 1 và toàn bộ artifact hiện có trên Google Drive được giữ nguyên. Notebook không xóa, chạy lại hoặc ghi đè Table 1. Sau khi ba run hoàn thành, notebook tự tạo CSV, PNG và provenance cho Table 2 rồi dừng.

## Trước khi chọn Runtime → Run all

1. Chọn GPU runtime.
2. Dừng mọi Colab session MyGram cũ.
3. Không xóa `MyDrive/MyGramRepro/runs/experiments`.

Nếu Colab bị ngắt, mở lại notebook và chọn **Runtime → Run all**. Các run completed sẽ được bỏ qua và run chưa xong sẽ resume từ snapshot Drive.


In [ ]:
import os
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
DRIVE_ROOT = Path('/content/drive/MyDrive/MyGramRepro')
BUNDLE = DRIVE_ROOT / 'source/mygram-source.zip'
BUNDLE_MANIFEST = DRIVE_ROOT / 'source/source_bundle_manifest.json'
PMF_ARCHIVE = DRIVE_ROOT / 'sources/pmf_mmkg.zip'
for required in (BUNDLE, BUNDLE_MANIFEST, PMF_ARCHIVE):
    if not required.is_file():
        raise FileNotFoundError(required)
print('Drive artifacts found:', DRIVE_ROOT)

In [ ]:
import hashlib, json, os, shutil, zipfile
manifest = json.loads(BUNDLE_MANIFEST.read_text(encoding='utf-8'))
def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()
if BUNDLE.stat().st_size != manifest['bundle']['bytes'] or sha256(BUNDLE) != manifest['bundle']['sha256']:
    raise ValueError('Source bundle does not match its manifest')
WORKSPACE = Path('/content/mygram')
if WORKSPACE.exists():
    shutil.rmtree(WORKSPACE)
WORKSPACE.mkdir()
with zipfile.ZipFile(BUNDLE) as archive:
    archive.extractall(WORKSPACE)
shutil.copy2(BUNDLE_MANIFEST, WORKSPACE / 'source_bundle_manifest.json')
os.chdir(WORKSPACE)
print('Source restored:', manifest['bundle']['sha256'])

In [ ]:
from src.notebook_runner import run_script
run_script(Path('scripts/notebook_steps/common/03_install_dependencies.py'), globals())


In [ ]:
run_script(Path('scripts/notebook_steps/common/04_check_environment.py'), globals())


In [ ]:
run_script(Path('scripts/notebook_steps/common/05_prepare_data.py'), globals())


## Apply the same checkpoint optimization used by the completed recent runs

In [ ]:
run_script(Path('scripts/notebook_steps/table_common/07_patch_checkpointing.py'), globals())


In [ ]:
run_script(Path('scripts/notebook_steps/table_common/08_record_patch_diff.py'), globals())


## Validate the patched source

In [ ]:
run_script(Path('scripts/notebook_steps/table_common/10_validate_patch.py'), globals())


## Clear a stale lock and run/resume only Table 2

In [ ]:
run_script(Path('scripts/notebook_steps/table_common/12_clear_stale_lock.py'), globals())


In [ ]:
run_script(Path('scripts/notebook_steps/table2/13_run_experiments.py'), globals())


## Build Table 2 automatically

In [ ]:
run_script(Path('scripts/notebook_steps/table2/15_build_report.py'), globals())


In [ ]:
run_script(Path('scripts/notebook_steps/table2/16_final_status.py'), globals())
